# Bokmål/nynorsk alignment av lånekassens dokumenter med LaBSE

In [40]:
import pandas as pd

df = pd.read_json("lånekassen_data.json")
df

,doc_hash,lang,url,domain,date,mimetype,fulltext
0,b5fc6f81d5709f0051da1964c3900e7ca8df65f7,nno,http://lanekassen.no/globalassets/brosjyrer-fe...,lanekassen.no,2022-12-19 01:53:28,pdf,[Er du flyktning? Du blir rekna som flyktning ...
1,1fbe096fdfdd9916be1313006ac5cf44ac650231,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:14,pdf,"[, , Du kan bruke dette skjemaet dersom du tar..."
2,582aaf9a510b3bde21b5b3e13ffc3453d6ca8174,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 01:57:21,pdf,"[, , Det er viktig at du les informasjonen på ..."
3,32e23cf05e4db850a9e2f622c2edd0482572677e,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:00:45,pdf,[Nynorsk Skjema for lærlinglønn Kor stort bort...
4,4ea46648c7ce59fe0c39b45779ed6402d8b41369,nno,http://lanekassen.no/globalassets/skjemaer-fel...,lanekassen.no,2022-12-19 02:01:01,pdf,[Nynorsk Skjema I – artikkelnr. 9001550 – nyno...
...,...,...,...,...,...,...,...
561,e29a43da7165d92d937c56416d50f563201ab5fa,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:18,pdf,"[, , 01.01.2020 Storebrand Bank ASA 70 % Bolig..."
562,c120ee7f582c758ae392b8f90795a7c664c39e90,nob,http://lanekassen.no/siteassets/skjemaer-og-fi...,lanekassen.no,2022-12-19 02:50:23,pdf,"[, , 06.11.2019 Storebrand Bank ASA 70 % Bolig..."
563,553101a9b0a9fac935cf31e5dd59555d23ecbefe,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:41,pdf,"[, , Flyktningstipendet Mottakere av flyktning..."
564,1f14c7bcc564613f79be7a8cae1a946e27cfd755,nob,https://statistikk.lanekassen.no/globalassets/...,lanekassen.no,2022-12-19 03:10:44,pdf,"[, , Tall og fakta om Lånekassens kunder og or..."


In [41]:
nynorske = df[df.lang == "nno"].copy()
nynorske.index = range(len(nynorske))

bokmålske = df[df.lang == "nob"].copy()
bokmålske.index = range(len(bokmålske))

len(nynorske), len(bokmålske)

(256, 310)

Lim sammen avsnittene i hvert dokument

In [42]:
nynorske["texts_joined"] = nynorske.fulltext.apply(lambda x: "\n".join(x))
bokmålske["texts_joined"] = bokmålske.fulltext.apply(lambda x: "\n".join(x))

Last inn fasit

In [43]:
hash_to_i_nn = {e.doc_hash: e.Index for e in nynorske.itertuples()}
hash_to_i_bm = {e.doc_hash: e.Index for e in bokmålske.itertuples()}

fasit = pd.read_csv("lanekassen_fasit.csv")
fasit_set = {(hash_to_i_nn[nn_doc_hash], hash_to_i_bm[bm_doc_hash]) for nn_doc_hash, bm_doc_hash in zip(fasit.nynorsk_doc_hash, fasit.bokmål_doc_hash)}

def compare_matches_to_fasit(matches):
    matches = {(i, match["corpus_id"]) for i, match in matches}

    hits = fasit_set.intersection(matches)
    misses = fasit_set - matches
    
    return (hits, misses)

Last inn modellen

In [44]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('sentence-transformers/LaBSE', device="cuda")

# Tell hvor mange dokumenter og avsnitt som er for lange for modellen

In [45]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

nynorsk_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in nynorske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in nynorsk_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(nynorsk_documents_token_sequence_lengths)} nynorske dokumentene er:
    {under_max_len} under LaBSE sin makslengde 
    {over_max_len}  over LaBSE sin makslengde 
Altså er {round(under_max_len/len(nynorsk_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")

bokmål_documents_token_sequence_lengths = [len(tokenizer.tokenize(text)) for text in bokmålske.texts_joined]

under_max_len = 0
over_max_len = 0
zero_len = 0
for token_len in bokmål_documents_token_sequence_lengths:
    if token_len:
        if token_len > max_len:
            over_max_len += 1
        else:
            under_max_len += 1
    else:
        zero_len += 1

assert zero_len == 0

print(f"""
Av de {len(bokmål_documents_token_sequence_lengths)} dokumentene på bokmål er:
    {under_max_len} under LaBSE sin makslengde 
    {over_max_len}  over LaBSE sin makslengde 
Altså er {round(under_max_len/len(bokmål_documents_token_sequence_lengths)*100, 2)}% av dokumentene under makslengden 
""")

Token indices sequence length is longer than the specified maximum sequence length for this model (1802 > 512). Running this sequence through the model will result in indexing errors



Av de 256 nynorske dokumentene er:
    48 under LaBSE sin makslengde 
    208  over LaBSE sin makslengde 
Altså er 18.75% av dokumentene under makslengden 


Av de 310 dokumentene på bokmål er:
    63 under LaBSE sin makslengde 
    247  over LaBSE sin makslengde 
Altså er 20.32% av dokumentene under makslengden 



In [46]:
max_len = model.max_seq_length
tokenizer = model.tokenizer

token_sequence_lengths = df.fulltext.apply(lambda x: [len(tokenizer.tokenize(e)) for e in x])

under_max_len = 0
over_max_len = 0
zero_len = 0
for e in token_sequence_lengths:
    for token_len in e:
        if token_len:
            if token_len > max_len:
                over_max_len += 1
            else:
                under_max_len += 1
        else:
            zero_len += 1

print(f"""
Det er totalt {sum((under_max_len, over_max_len))} ikke-tomme avsnitt/setninger (og {zero_len} er tomme)
Av de ikke-tomme avsnittene er:
    {under_max_len} under LaBSE sin makslengde 
    {over_max_len}  over LaBSE sin makslengde 
Altså er {round(under_max_len/sum((under_max_len, over_max_len))*100, 2)}% av de ikke-tomme avsnittene under makslengden 
""")



Det er totalt 12491 ikke-tomme avsnitt/setninger (og 4377 er tomme)
Av de ikke-tomme avsnittene er:
    11648 under LaBSE sin makslengde 
    843  over LaBSE sin makslengde 
Altså er 93.25% av de ikke-tomme avsnittene under makslengden 



# Dokumentalignment
Finn den likeste bokmål-dokument-embeddingen for hver nynorsk-dokument-embedding.  

In [47]:
from pathlib import Path 

def write_doc_matches_to_file(matches, filename):
    file_path = Path(filename)
    file_path.parent.mkdir(exist_ok=True, parents=True)
    
    nn_i, bm_i = zip(*[(i, res["corpus_id"]) for i, res in matches])
                    
    nn_hashes = list(nynorske.doc_hash.iloc[list(nn_i)])
    bm_hashes = list(bokmålske.doc_hash.iloc[list(bm_i)])

    pd.DataFrame({"nynorsk_doc_hash": nn_hashes, "bokmål_doc_hash": bm_hashes}).to_csv(filename, index=False)

In [48]:
doc_alignment_results = {}

## Aksepter cut-off

Send dokumentet as is til modellen (vil kuttes av på modellens makslengde)

In [49]:
import numpy as np 
from pathlib import Path

base_path = "labse/texts_joined"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    bokmål_embeddings = model.encode(bokmålske.texts_joined)
    nynorsk_embeddings = model.encode(nynorske.texts_joined)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")


hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["naiv_cutoff"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

### Inspiser resultater 

In [50]:
from utils import print_matches, print_misses

# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

#### Falsk positiv
Eksempel på en falsk positiv.  
Noen lister av datoer og banker og renter blir veldig like  

In [51]:
i = 21
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 293, 'score': 0.9657115936279297}]


01.09.2021 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,29 % 01.09.2021 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1,33 % 01.09.2021 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,47 % 01.09.2021 OBOS-banken AS B

___________



03.11.2021 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,29 % 03.11.2021 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,47 % 03.11.2021 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1,56 % 03.11.2021 Sunndal Spareban


#### Falsk negativ

In [52]:
i = 80
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 46, 'score': 0.9480304718017578}]
Personvern­erklæring
Kva er personopplysningar?
Ei personopplysning er ei opplysning som er med på å identifisere deg som enkeltperson. Det blir skilt mellom personopplysningar og særlege kategoriar (sensitive) personopplysningar.
Lånekassen er behan

___________

Personvern­erklæring
Hva er personopplysninger?
En personopplysning er en opplysning som er med på å identifisere deg som enkeltperson. Det skilles mellom personopplysninger og særlige kategorier (sensitive) personopplysninger.
Lånekassen er behandli


## Del opp dokumentene i biter mindre enn modellens makslengde og aggreger

In [53]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in nynorske.texts_joined]
bokmål_maxlen_parts = [tokenize_and_split_text(text, tokenizer=model.tokenizer, model_max_len=model.max_seq_length) for text in bokmålske.texts_joined]

In [54]:
# from collections import Counter 
# pd.DataFrame(Counter([len(e) for e in nynorsk_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")
# pd.DataFrame(Counter([len(e) for e in bokmål_maxlen_parts]).most_common(), columns=("antall lister", "antall dokumenter")).sort_values("antall lister")

### Mean pooling

In [55]:
import numpy as np 

base_path = "labse/maxlen_parts_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.mean(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")

hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["mean_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

#### Inspiser resultater

In [56]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

In [57]:
for nn_i, bm_i in misses:
    match_i = search_result[nn_i][0]["corpus_id"]
    match_score = search_result[nn_i][0]["score"]
    
    if match_i != bm_i:
        print("Likeste søketreff er et annet dokument enn fasit\n")
        print(f"Fasit indeks: {bm_i}\nTreff indeks: {match_i}")
        print(f"Søketekst og fasit likhet: {float(util.cos_sim(nynorsk_embeddings[nn_i], bokmål_embeddings[bm_i]))}")
        print(f"Søketekst og match likhet: {match_score}")
        print(f"Treff og fasit likhet {float(util.cos_sim(bokmål_embeddings[bm_i], bokmål_embeddings[match_i]))}\n\n")

        print(f"Nynorsk søketekst:\n\t{nynorske.texts_joined[nn_i][:300]}\n_____")
        print(f"Bokmål søketreff:\n\t{bokmålske.texts_joined[match_i][:300]}\n_____")
        print(f"Bokmål fasit:\n\t{bokmålske.texts_joined[bm_i][:300]}\n_____")

    else:
        print("Likeste søketreff er det samme som fasiten, men similarity score var under terskelen\n")
        assert match_score <= threshold

Likeste søketreff er det samme som fasiten, men similarity score var under terskelen

Likeste søketreff er et annet dokument enn fasit

Fasit indeks: 74
Treff indeks: 59
Søketekst og fasit likhet: 0.8689754009246826
Søketekst og match likhet: 0.8964160680770874
Treff og fasit likhet 0.9152108430862427


Nynorsk søketekst:
	Søknad om lån og stipend til nettstudiar ved ein norsk lærestad
Dette søker du om
I søknaden søker du om basislån og eventuelt skolepengelån. Du kan velje å søke om berre den delen av basislånet som kan bli gjort om til stipend, eller heile lånet.
Søknaden gjeld også for andre stipend og lån som ber
_____
Bokmål søketreff:
	Søknad for fagskole
Dette søker du om
I søknaden søker du om basislån og eventuelt skolepengelån. Du kan velge å søke om bare den delen av basislånet som kan gjøres om til stipend, eller hele lånet.
Søknaden gjelder også for andre stipend og lån som bare noen har krav på. Dette er barnestipend, till
_____
Bokmål fasit:
	Søknad om lån og stipend ti

##### Falsk positiv:

In [58]:
i = 19
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 291, 'score': 0.9716790318489075}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



02.03.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1.79% 02.03.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1.81% 02.03.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1.97% 02.03.2022 KLP Banken AS Bolig


##### Falsk negativ

In [59]:
i = 72
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 38, 'score': 0.936010479927063}]
Hjelp og kontakt
Veileder frister
Aktuelt
Nytt straumstipend hausten 2022
Alle som kan ha rett til strømstipend, har fått ein e-post frå oss. Logg inn for å søke. Søknadsfristen er 15. januar 2023.
Bruker du avtalegiro? Sjekk nettbanken
06.12.2022
På

___________

Hjelp og kontakt
Veileder frister
Aktuelt
Nytt strømstipend høsten 2022
Alle som kan ha rett til strømstipend, har fått en e-post fra oss. Logg inn for å søke. Søknadsfristen er 15. januar 2023.
Bruker du avtalegiro? Sjekk nettbanken
06.12.2022
På gr


### Max pooling

In [60]:
import numpy as np 

base_path = "labse/maxlen_parts_max_pooling"
emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in nynorsk_maxlen_parts])
    bokmål_embeddings = np.array([np.max(model.encode(text_parts), axis=0) for text_parts in bokmål_maxlen_parts])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)
threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]
write_doc_matches_to_file(matches, f"output/{base_path}.csv")


hits, misses = compare_matches_to_fasit(matches)
doc_alignment_results["max_pooling_biter"] = {"matches": len(matches), "threshold": threshold, "percent of docs": round(len(matches)/len(nynorske)*100, 2), "hits": len(hits), "misses": len(misses)}

#### Inspiser resultater

In [61]:
# print_matches(matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_matches(non_matches, nynorske.texts_joined, bokmålske.texts_joined)
# print_misses(misses, search_result, bokmålske.texts_joined, nynorske.texts_joined, bokmål_embeddings, nynorsk_embeddings, threshold)

##### Falsk positiv

In [62]:
i = 19
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 291, 'score': 0.937472939491272}]


04.05.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1,79% 04.05.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1,97% 04.05.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 2,07% 04.05.2022 Sunndal Sparebank B

___________



02.03.2022 Bulder Bank (Sparebanken Vest) Boliglån innenfor 50 % 1.79% 02.03.2022 Himla Banktjenester (Fana Sparebank) Boliglån innenfor 75 % 1.81% 02.03.2022 NORDirekte (Skagerrak Sparebank) Boliglån inntil 50% 1.97% 02.03.2022 KLP Banken AS Bolig


##### Falske negativer
Veldig lav score!

In [63]:
i = 72
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 38, 'score': 0.7574380040168762}]
Hjelp og kontakt
Veileder frister
Aktuelt
Nytt straumstipend hausten 2022
Alle som kan ha rett til strømstipend, har fått ein e-post frå oss. Logg inn for å søke. Søknadsfristen er 15. januar 2023.
Bruker du avtalegiro? Sjekk nettbanken
06.12.2022
På

___________

Hjelp og kontakt
Veileder frister
Aktuelt
Nytt strømstipend høsten 2022
Alle som kan ha rett til strømstipend, har fått en e-post fra oss. Logg inn for å søke. Søknadsfristen er 15. januar 2023.
Bruker du avtalegiro? Sjekk nettbanken
06.12.2022
På gr


In [64]:
i = 80
print(search_result[i])
print(nynorske.texts_joined[i][:250])
print("\n___________\n")
print(bokmålske.texts_joined[search_result[i][0]["corpus_id"]][:250])

[{'corpus_id': 46, 'score': 0.919640064239502}]
Personvern­erklæring
Kva er personopplysningar?
Ei personopplysning er ei opplysning som er med på å identifisere deg som enkeltperson. Det blir skilt mellom personopplysningar og særlege kategoriar (sensitive) personopplysningar.
Lånekassen er behan

___________

Personvern­erklæring
Hva er personopplysninger?
En personopplysning er en opplysning som er med på å identifisere deg som enkeltperson. Det skilles mellom personopplysninger og særlige kategorier (sensitive) personopplysninger.
Lånekassen er behandli


## Konklusjon

Vi får like mange treff på fasit med naiv cutoff og mean pooling.  
Men naiv cutoff treffer litt flere av den totale mengden dokumenter.  
Vi har ikke gulldata å sammenlikne med, så det er ikke så godt å si helt sikkert om dette er den beste metoden.

In [65]:
pd.DataFrame(doc_alignment_results).T.sort_values("hits")

,matches,threshold,percent of docs,hits,misses
max_pooling_biter,120.0,0.95,46.88,26.0,29.0
naiv_cutoff,217.0,0.95,84.77,48.0,7.0
mean_pooling_biter,212.0,0.95,82.81,48.0,7.0


# Setnings/avsnittsalignment

In [66]:
sent_alignment_results = {}

In [67]:
from collections import defaultdict

nynorske_sentences = defaultdict(list)
for t, df_ in nynorske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        nynorske_sentences["text"].append(t)
        nynorske_sentences["doc_hashes"].append(set(df_.doc_hash))
        nynorske_sentences["urls"].append(set(df_.url))

nynorske_flat = pd.DataFrame(nynorske_sentences)

bokmålske_sentences = defaultdict(list)
for t, df_ in bokmålske.explode("fulltext").groupby("fulltext"):
    if len(t)>2:
        bokmålske_sentences["text"].append(t)
        bokmålske_sentences["doc_hashes"].append(set(df_.doc_hash))
        bokmålske_sentences["urls"].append(set(df_.url))

bokmålske_flat = pd.DataFrame(bokmålske_sentences)

In [68]:
len(bokmålske_flat), len(nynorske_flat)

(6345, 4026)

In [69]:
def write_sent_matches_to_file(matches, filename):
    nn_i, bm_i = zip(*[(i, res["corpus_id"]) for i, res in matches])

    nn = nynorske_flat.iloc[list(nn_i)].rename(mapper=lambda x: "nn_"+x, axis=1)
    bm = bokmålske_flat.iloc[list(bm_i)].rename(mapper=lambda x: "bm_"+x, axis=1)
    nn.index = range(len(nn))
    bm.index = range(len(bm))
    
    pd.concat([nn, bm], axis=1).to_csv(filename, index=False)


In [70]:
same_text = bokmålske_flat.merge(nynorske_flat, on="text", suffixes=["_bm", "_nn"])
sent_alignment_results["string_comparison"] = {"matches": len(same_text), "percent of sents": round(len(same_text)/len(nynorske_flat), 2)}

## Aksepter cut-off
Godta cut-off på LABSE sin maxlengde

In [71]:
import numpy as np

base_path = "labse/texts_flat"

emb_path = Path(f"embeddings/{base_path}.npz")
if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = model.encode(nynorske_flat.text)
    bokmål_embeddings = model.encode(bokmålske_flat.text)
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["naiv_cutoff"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorske_flat.text)*100, 2)}

### Inspiser resultater

In [72]:
from utils import print_matches
 
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=30)

## Del opp setninger/avsnitt som er lengre enn LABSE sin maxlengde og agregger

In [73]:
from utils import tokenize_and_split_text

nynorsk_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in nynorske_flat.text]
bokmål_maxlen_parts_flat = [tokenize_and_split_text(text, model.tokenizer, model.max_seq_length) for text in bokmålske_flat.text]

### Mean pooling

In [74]:
base_path = "/labse/maxlen_parts_flat_mean_pooling"

emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.mean(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["mean_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

#### Inspiser resultater

In [75]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)


### Max pooling

In [76]:
base_path = "labse/maxlen_parts_flat_max_pooling"
emb_path = Path(f"embeddings/{base_path}.npz")

if emb_path.exists():
    loaded = np.load(emb_path)
    bokmål_embeddings = loaded["a"]
    nynorsk_embeddings = loaded["b"]
else:
    emb_path.parent.mkdir(exist_ok=True, parents=True)
    nynorsk_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in nynorsk_maxlen_parts_flat])
    bokmål_embeddings = np.array([np.max(model.encode(sent), axis=0) for sent in bokmål_maxlen_parts_flat])
    np.savez_compressed(emb_path, a=bokmål_embeddings, b=nynorsk_embeddings)

search_result = util.semantic_search(nynorsk_embeddings, bokmål_embeddings, top_k=1)

threshold = 0.95
matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] > threshold]
non_matches = [(i, e[0]) for i, e in enumerate(search_result) if e[0]["score"] <= threshold]

write_sent_matches_to_file(matches, f"output/{base_path}.csv")
sent_alignment_results["max_pooling_biter"] = {"threshold": threshold, "matches": len(matches), "percent of sents": round(len(matches)/len(nynorsk_maxlen_parts_flat)*100, 2)}

#### Inspiser resultater

In [77]:
# print_matches(matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)
# print_matches(non_matches, nynorske_flat.text, bokmålske_flat.text, stop_printing_at=50)


## Konklusjon
Vi får flest matches med naiv cutoff (vi vet at over 93% av avsnittene er innenfor modellens makslengde), etterfulgt av mean pooling.  
Vi har ikke gulldata å sammenlikne med, så det er ikke så godt å si helt sikkert om dette er den beste metoden.

In [78]:
pd.DataFrame(sent_alignment_results).T.sort_values("matches")

,matches,percent of sents,threshold
string_comparison,338.0,0.08,NaN
max_pooling_biter,2708.0,67.26,0.95
mean_pooling_biter,2720.0,67.56,0.95
naiv_cutoff,2725.0,67.69,0.95
